# Module 10A Integrated Lab — Four-Service Movie Information API

**Session anchor:** 110 minutes  
**Stack:** FastAPI · PostgreSQL · ChromaDB · Next.js  
**Orchestration tool:** Docker Compose  

---

| Beat | Topic | Time |
|------|-------|------|
| 1 | Setup orientation & recap | 5 min |
| 2 | Two-service skeleton — backend + Postgres | 15 min |
| 3 | Adding the vector store — Chroma | 15 min |
| 4 | Adding the frontend service | 12 min |
| 5 | Healthcheck + depends_on chain | 15 min |
| 6 | .env discipline | 8 min |
| 7 | Seed orchestration | 15 min |
| 8 | End-to-end demo + partial-failure recovery | 10 min |
| 9 | Runbook authoring | 10 min |
| 10 | Team rollout & assignment pivot | 10 min |

> **Total: 115 min** — within ±10 % of the 110-min session anchor.

---

**Pre-identified artifacts required before session start:**
- `seed_postgres.sql` — 20 movie rows (generated in Beat 2)
- `movie_reviews.json` — 30 review chunks for Chroma (generated in Beat 3)
- Slide: *build-time vs runtime env in Next.js* (referenced in Beat 4)
- Terminal capture: failing-then-passing startup (referenced in Beat 5)
- `.env` checker script (generated in Beat 6)

## Beat 1 — Setup Orientation & Recap (5 min)

### Day 2 Architecture
In Day 2 we built and tagged two Docker images:

| Image build context | Purpose |
|--------------------|---------|
| `./api` | FastAPI: movie routes, Postgres queries, Chroma RAG |
| `./web` | Next.js: user-facing search UI |

### Today — Four-Service Composition

```
┌──────────────────────────────────────────────────────────┐
│                   Docker Compose Network                 │
│                                                          │
│  ┌─────────────────────────────────────────────────┐    │
│  │  web  (Next.js · port 3000)                     │    │
│  │  depends_on: api (service_healthy)               │    │
│  └───────────────────┬─────────────────────────────┘    │
│                      │ HTTP                              │
│  ┌───────────────────▼─────────────────────────────┐    │
│  │  api  (FastAPI · port 8080)                     │    │
│  │  depends_on: db (healthy) + chroma (healthy)    │    │
│  └──────┬───────────────────────┬──────────────────┘    │
│         │ SQL                   │ HTTP                   │
│  ┌──────▼──────┐       ┌───────▼────────┐              │
│  │  db         │       │  chroma        │              │
│  │  postgres   │       │  chromadb      │              │
│  │  port 5432  │       │  port 8000     │              │
│  └─────────────┘       └────────────────┘              │
└──────────────────────────────────────────────────────────┘
```

### Build order
We add one service per beat so each Compose concept lands before the next one arrives.  
Beat 2 → `api + db` | Beat 3 → `+ chroma` | Beat 4 → `+ web` | Beat 5 → healthchecks

In [1]:
import subprocess                                                # runs shell commands from Python
import sys                                                       # access to Python interpreter info
import os                                                        # OS-level file and directory operations
import json                                                      # JSON serialization and deserialization
import time                                                      # time-related sleep and timing utilities

def run(cmd, capture=True, echo=True):                           # helper: runs a shell command and returns result
    if echo:                                                     # optionally prints the command before running
        print(f"$ {cmd}")                                        # shows the command string to the user
    result = subprocess.run(                                     # executes the command in a child process
        cmd, shell=True,                                         # shell=True allows full shell syntax
        capture_output=capture,                                  # captures both stdout and stderr streams
        text=True                                                # decodes output bytes as UTF-8 text
    )                                                            # returns a CompletedProcess instance
    if capture and result.stdout:                                # only print stdout if there is content
        print(result.stdout.strip())                             # strips trailing whitespace before printing
    if capture and result.stderr and result.returncode != 0:    # only print stderr on non-zero exit code
        print(result.stderr.strip(), file=sys.stderr)           # writes error message to the stderr stream
    return result                                                # returns the CompletedProcess for inspection

print("=== Prerequisite Check ===\n")                           # section header for this cell

r = run("docker --version")                                      # checks whether Docker CLI is installed
print("✓ Docker found" if r.returncode == 0 else "✗ Install Docker Desktop first")

r = run("docker compose version")                               # checks for the Compose v2 plugin
print("✓ Compose v2 found" if r.returncode == 0 else "✗ Compose v2 not found")

for img in ["postgres:15-alpine", "chromadb/chroma:latest"]:    # iterates over images that must be pre-pulled
    r = run(f"docker image inspect {img} -f '{{{{.Id}}}}'" )    # inspects local image, fails if not present
    status = "✓" if r.returncode == 0 else f"✗  Run: docker pull {img}"  # marks pass or print pull command
    print(f"{status}  {img}")                                    # prints the check result per image

for d in ["api", "web/pages"]:                                   # iterates over required sub-directories
    os.makedirs(d, exist_ok=True)                                # creates directory tree, skips if it exists
print("\n✓ Directory scaffold ready.")                           # confirms that api/ and web/ directories exist

=== Prerequisite Check ===

$ docker --version
Docker version 29.1.3, build 29.1.3-0ubuntu3~22.04.2
✓ Docker found
$ docker compose version
Docker Compose version v2.25.0
✓ Compose v2 found
$ docker image inspect postgres:15-alpine -f '{{.Id}}'
sha256:5cce759a2777634ff1edd0d56b9241a2961deb7f72e125d4af6ae2928163a6b6
✓  postgres:15-alpine
$ docker image inspect chromadb/chroma:latest -f '{{.Id}}'
sha256:a221f3391b5eabc5b0b293b65958a28f10d7015be67e4d8cc33a5e4e60e6f3ca
✓  chromadb/chroma:latest

✓ Directory scaffold ready.


## Beat 2 — Two-Service Skeleton: Backend + Postgres (15 min)

We start with the smallest viable composition: `api` and `db`.

### Key decisions
| Decision | Reason |
|----------|--------|
| `postgres:15-alpine` | Small, stable, LTS-track Postgres image |
| Named volume `pgdata` | Data survives `docker compose down`; destroyed only with `-v` |
| Port `5432` published | Lets psql / pgAdmin connect from the host during development |
| Seed via `initdb.d/` | Postgres auto-runs every `.sql` file in that directory on first boot |

### Pre-identified artifact
`seed_postgres.sql` — 20 movies written into `initdb.d/` at container start.

In [2]:
%%writefile api/requirements.txt
fastapi==0.111.0
uvicorn[standard]==0.30.1
psycopg2-binary==2.9.9
chromadb>=1.4.1
pydantic==2.7.1
httpx==0.27.0

Overwriting api/requirements.txt


In [ ]:
%%writefile api/Dockerfile
# ── Build stage ──────────────────────────────────────────────
# official slim Python 3.11 image as the base
FROM python:3.11-slim AS base

# installs curl for the /readyz healthcheck probe
RUN apt-get update && apt-get install -y --no-install-recommends curl && rm -rf /var/lib/apt/lists/*

# sets /app as the working directory inside the image
WORKDIR /app

# copies only the dependency list first (layer-cache optimisation)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt  # installs all Python packages without caching

# copies the rest of the application source code
COPY . .

# documents the port the app will listen on (informational)
EXPOSE 8080

# starts FastAPI with uvicorn
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]

In [ ]:
%%writefile api/main.py
"""Movie Information API — FastAPI backend for Module 10A."""
from fastapi import FastAPI, HTTPException, Query   # FastAPI core and helpers
from fastapi.middleware.cors import CORSMiddleware  # allows browser cross-origin requests
import psycopg2                                     # PostgreSQL driver
import psycopg2.extras                              # provides RealDictCursor for dict-style rows
import chromadb                                     # ChromaDB Python client
import os                                           # reads environment variables

app = FastAPI(title="Movie API", version="1.0.0")   # creates the FastAPI application instance

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # permit requests from the Next.js frontend on any origin
    allow_methods=["GET"],
    allow_headers=["*"],
)

def get_pg_conn():                                  # factory: returns a new Postgres connection
    return psycopg2.connect(os.environ["DATABASE_URL"])  # reads DATABASE_URL from environment

def get_chroma():                                   # factory: returns a ChromaDB HTTP client
    return chromadb.HttpClient(                     # connects to Chroma over HTTP
        host=os.environ.get("CHROMA_HOST", "localhost"),   # hostname injected by Compose
        port=int(os.environ.get("CHROMA_PORT", "8000"))    # port injected by Compose
    )

@app.get("/readyz")                                 # readiness probe endpoint — checked by healthcheck
def readyz():
    try:                                            # attempts to connect to Postgres
        conn = get_pg_conn()                        # opens a Postgres connection
        conn.close()                                # closes connection immediately after check
    except Exception as exc:                        # catches any connection failure
        raise HTTPException(503, detail=f"Postgres not ready: {exc}")  # returns 503 to caller
    try:                                            # attempts to connect to Chroma
        get_chroma().heartbeat()                    # Chroma heartbeat raises if unreachable
    except Exception as exc:                        # catches any Chroma connection failure
        raise HTTPException(503, detail=f"Chroma not ready: {exc}")    # returns 503 to caller
    return {"status": "ok"}                         # both dependencies healthy — return 200

@app.get("/movies")                                 # endpoint: returns all movies from Postgres
def list_movies():
    conn = get_pg_conn()                            # opens a fresh Postgres connection
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:  # dict-style rows
        cur.execute("SELECT id, title, genre, year, director, rating FROM movies ORDER BY rating DESC")  # queries all movies sorted by rating
        rows = cur.fetchall()                       # fetches all result rows into memory
    conn.close()                                    # closes connection to return it to the pool
    return [{**r} for r in rows]                   # converts RealDictRow objects to plain dicts

@app.get("/search")                                 # endpoint: semantic search via Chroma RAG
def search_reviews(q: str = Query(..., min_length=1)):  # q is a required query-string parameter
    client = get_chroma()                           # creates a fresh Chroma HTTP client
    try:                                            # wraps Chroma call to catch collection errors
        collection = client.get_collection("movie_reviews")  # gets existing collection by name
    except Exception:                               # collection missing means Chroma not seeded yet
        raise HTTPException(503, detail="movie_reviews collection not seeded")  # 503 until seeded
    results = collection.query(                     # performs approximate-nearest-neighbour search
        query_texts=[q],                            # embeds the query string automatically
        n_results=3                                 # returns the 3 most similar review chunks
    )
    return {                                        # returns structured JSON response
        "query": q,                                 # echoes the search query
        "results": results["documents"][0],          # list of matching review text chunks
        "metadatas": results["metadatas"][0],        # list of metadata dicts (movie, year, sentiment)
    }

In [ ]:
compose_step1 = """version: \"3.9\"            # Compose file-format version

services:

  db:                              # PostgreSQL relational store
    image: postgres:15-alpine      # official Postgres 15 on Alpine Linux (small footprint)
    restart: unless-stopped        # restarts automatically unless explicitly stopped
    environment:
      POSTGRES_DB: ${POSTGRES_DB:-movies}           # database name; default 'movies'
      POSTGRES_USER: ${POSTGRES_USER:-movie_user}   # db user; default 'movie_user'
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-secret}  # password; override in .env
    volumes:
      - pgdata:/var/lib/postgresql/data            # named volume persists data across restarts
      - ./seed_postgres.sql:/docker-entrypoint-initdb.d/seed.sql:ro  # auto-seeds on first boot

  api:                             # FastAPI backend
    build: ./api                   # builds from the Dockerfile in ./api
    restart: unless-stopped
    ports:
      - \"8080:8080\"              # publishes FastAPI port to host
    environment:
      DATABASE_URL: postgresql://${POSTGRES_USER:-movie_user}:${POSTGRES_PASSWORD:-secret}@db:5432/${POSTGRES_DB:-movies}
      CHROMA_HOST: chroma          # Compose service name resolves on the internal network
      CHROMA_PORT: \"8000\"

volumes:
  pgdata:                          # declares the named volume used by the db service
"""

with open("docker-compose.yml", "w") as f:           # opens docker-compose.yml for writing
    f.write(compose_step1)                            # writes Step-1 Compose definition to disk

print("✓ docker-compose.yml written (Step 1: api + db)")  # confirms the file was written

In [6]:
%%writefile seed_postgres.sql
-- Movie catalog seed — 20 rows
-- This file is mounted into /docker-entrypoint-initdb.d/ and runs automatically on first boot.

CREATE TABLE IF NOT EXISTS movies (    -- creates the table only if it does not already exist
    id        SERIAL PRIMARY KEY,      -- auto-incrementing integer primary key
    title     VARCHAR(255) NOT NULL,   -- movie title, required field
    genre     VARCHAR(100),            -- genre label (nullable)
    year      INTEGER,                 -- release year (nullable)
    director  VARCHAR(255),            -- director full name (nullable)
    rating    DECIMAL(3,1)             -- audience rating on a 10-point scale
);

INSERT INTO movies (title, genre, year, director, rating) VALUES
  ('The Shawshank Redemption',                    'Drama',   1994, 'Frank Darabont',          9.3),
  ('The Godfather',                               'Crime',   1972, 'Francis Ford Coppola',    9.2),
  ('The Dark Knight',                             'Action',  2008, 'Christopher Nolan',       9.0),
  ('Pulp Fiction',                                'Crime',   1994, 'Quentin Tarantino',       8.9),
  ('Schindler''s List',                           'History', 1993, 'Steven Spielberg',        8.9),
  ('The Lord of the Rings: The Return of the King','Fantasy',2003, 'Peter Jackson',           8.9),
  ('Forrest Gump',                                'Drama',   1994, 'Robert Zemeckis',         8.8),
  ('Fight Club',                                  'Drama',   1999, 'David Fincher',           8.8),
  ('Inception',                                   'Sci-Fi',  2010, 'Christopher Nolan',       8.8),
  ('The Matrix',                                  'Sci-Fi',  1999, 'Lana Wachowski',          8.7),
  ('Goodfellas',                                  'Crime',   1990, 'Martin Scorsese',         8.7),
  ('The Silence of the Lambs',                    'Thriller',1991, 'Jonathan Demme',          8.6),
  ('Interstellar',                                'Sci-Fi',  2014, 'Christopher Nolan',       8.6),
  ('Saving Private Ryan',                         'War',     1998, 'Steven Spielberg',        8.6),
  ('The Green Mile',                              'Drama',   1999, 'Frank Darabont',          8.6),
  ('Parasite',                                    'Thriller',2019, 'Bong Joon-ho',            8.5),
  ('Whiplash',                                    'Drama',   2014, 'Damien Chazelle',         8.5),
  ('Dune',                                        'Sci-Fi',  2021, 'Denis Villeneuve',        8.0),
  ('Everything Everywhere All at Once',           'Sci-Fi',  2022, 'Daniels',                 7.8),
  ('Oppenheimer',                                 'History', 2023, 'Christopher Nolan',       8.3)
ON CONFLICT DO NOTHING;              -- idempotent: re-running will not duplicate rows

Overwriting seed_postgres.sql


In [7]:
print("Starting Step-1 stack (api + db)...")            # informs user which services will start

result = run("docker compose up --build -d")             # builds api image then starts both services detached

time.sleep(8)                                            # waits 8 s for Postgres to finish initialising

result = run("docker compose ps")                        # lists the running containers and their status

print("\nTip: run 'docker compose logs -f db' to watch Postgres init logs.")  # points to useful log command

Starting Step-1 stack (api + db)...
$ docker compose up --build -d
#0 building with "default" instance using docker driver

#1 [api internal] load build definition from Dockerfile
#1 DONE 0.0s

#1 [api internal] load build definition from Dockerfile
#1 transferring dockerfile:
#1 transferring dockerfile: 764B 0.2s done
#1 DONE 2.3s

#2 [api internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 10.1s

#3 [api internal] load .dockerignore
#3 transferring context:
#3 transferring context: 2B done
#3 DONE 1.8s

#4 [api internal] load build context
#4 DONE 0.0s

#5 [api 1/5] FROM docker.io/library/python:3.11-slim@sha256:cdbd05fb6f457ca275ff51ce00d93d865ca0b6a25f5ffb08262d94f6835771e5
#5 resolve docker.io/library/python:3.11-slim@sha256:cdbd05fb6f457ca275ff51ce00d93d865ca0b6a25f5ffb08262d94f6835771e5
#5 resolve docker.io/library/python:3.11-slim@sha256:cdbd05fb6f457ca275ff51ce00d93d865ca0b6a25f5ffb08262d94f6835771e5 3.0s done
#5 DONE 3.0s

#4 [api internal] load build conte

time="2026-06-25T05:48:15+03:00" level=warning msg="/home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 10A Integrated Lab/docker-compose.yml: `version` is obsolete"
time="2026-06-25T05:53:03+03:00" level=warning msg="Found orphan containers ([module10aintegratedlab-web-1 module10aintegratedlab-chroma-1]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."
 Container module10aintegratedlab-api-1  Recreate
 Container module10aintegratedlab-db-1  Recreate
 Container module10aintegratedlab-api-1  Recreated
 Container module10aintegratedlab-db-1  Recreated
 Container module10aintegratedlab-db-1  Starting
 Container module10aintegratedlab-api-1  Starting
 Container module10aintegratedlab-api-1  Started
Error response from daemon: failed to set up container networking: driver failed programming external connectivity on endpoint module10aintegratedlab-db-1 (0b05a06d256bed0908a5e67f

$ docker compose ps
NAME                              IMAGE                        COMMAND                  SERVICE   CREATED          STATUS                   PORTS
module10aintegratedlab-api-1      module10aintegratedlab-api   "uvicorn main:app --…"   api       36 seconds ago   Up 18 seconds            0.0.0.0:8080->8080/tcp, :::8080->8080/tcp
module10aintegratedlab-chroma-1   chromadb/chroma:latest       "dumb-init -- chroma…"   chroma    5 hours ago      Up 5 hours (unhealthy)   0.0.0.0:8000->8000/tcp, :::8000->8000/tcp

Tip: run 'docker compose logs -f db' to watch Postgres init logs.


## Beat 3 — Adding the Vector Store: Chroma (15 min)

We extend `docker-compose.yml` with the `chroma` service.

### Why a healthcheck is critical for Chroma
Chroma initialises its on-disk index after the container is marked *running*.  
The `api` **must not start** until Chroma's HTTP server responds — otherwise the first
embedding query will hit a connection-refused error before the index is ready.

Healthcheck command:
```bash
curl -sf http://localhost:8000/api/v1/heartbeat
```
- `-s` = silent (no progress bar)
- `-f` = fail with exit code 22 on HTTP 4xx/5xx

### Pre-identified artifact
`movie_reviews.json` — 30 review chunks; written to disk in the next cell and seeded into Chroma in Beat 7.

In [8]:
reviews = [                                                      # list of 30 movie-review chunks
    {"id": "rev_001", "text": "The Shawshank Redemption is a timeless tale of hope. Andy Dufresne's quiet determination behind bars inspires everyone around him, including the cynical Red.",                                                                 "metadata": {"movie": "The Shawshank Redemption",            "year": 1994, "sentiment": "positive"}},
    {"id": "rev_002", "text": "A devastating critique of prison brutality, Shawshank never loses its warmth. The friendship between Andy and Red is the emotional spine of the film.",                                                                          "metadata": {"movie": "The Shawshank Redemption",            "year": 1994, "sentiment": "positive"}},
    {"id": "rev_003", "text": "The Godfather redefined what a crime film could be. Brando's performance as Vito Corleone is measured, powerful, and deeply human.",                                                                                           "metadata": {"movie": "The Godfather",                        "year": 1972, "sentiment": "positive"}},
    {"id": "rev_004", "text": "Coppola's masterwork depicts the corruption of the American dream through the Corleone family's tragic descent into violence and power.",                                                                                        "metadata": {"movie": "The Godfather",                        "year": 1972, "sentiment": "positive"}},
    {"id": "rev_005", "text": "The Dark Knight is the rare superhero film that transcends the genre. Ledger's Joker is a force of chaos that feels genuinely threatening.",                                                                                    "metadata": {"movie": "The Dark Knight",                      "year": 2008, "sentiment": "positive"}},
    {"id": "rev_006", "text": "Nolan's Gotham is grimy and credible. The Dark Knight plays more like a crime thriller than a comic-book adaptation.",                                                                                                         "metadata": {"movie": "The Dark Knight",                      "year": 2008, "sentiment": "positive"}},
    {"id": "rev_007", "text": "Pulp Fiction shattered narrative convention. Tarantino's non-linear structure keeps you disoriented in the best possible way.",                                                                                                "metadata": {"movie": "Pulp Fiction",                         "year": 1994, "sentiment": "positive"}},
    {"id": "rev_008", "text": "The dialogue in Pulp Fiction is unlike anything before or since — playful, menacing, and full of pop-culture riffs that never feel dated.",                                                                                    "metadata": {"movie": "Pulp Fiction",                         "year": 1994, "sentiment": "positive"}},
    {"id": "rev_009", "text": "Schindler's List is an almost unbearable act of bearing witness. Spielberg's restraint — most notably the use of black and white — makes the horror more real.",                                                               "metadata": {"movie": "Schindler's List",                     "year": 1993, "sentiment": "emotional"}},
    {"id": "rev_010", "text": "Liam Neeson's transformation from profiteer to rescuer is one of cinema's great redemption arcs. Schindler's List demands to be seen.",                                                                                       "metadata": {"movie": "Schindler's List",                     "year": 1993, "sentiment": "positive"}},
    {"id": "rev_011", "text": "Inception is a lucid dream about the mechanics of storytelling itself. Each dream layer mirrors a layer of Nolan's own cinematic construction.",                                                                               "metadata": {"movie": "Inception",                           "year": 2010, "sentiment": "positive"}},
    {"id": "rev_012", "text": "Hans Zimmer's slowed-down Edith Piaf sample in Inception's score is one of the great musical jokes in film history — and most audiences never catch it.",                                                                     "metadata": {"movie": "Inception",                           "year": 2010, "sentiment": "positive"}},
    {"id": "rev_013", "text": "Fight Club is a seductive trap. It invites you to root for Tyler Durden before revealing how poisonous his philosophy really is.",                                                                                            "metadata": {"movie": "Fight Club",                          "year": 1999, "sentiment": "mixed"}},
    {"id": "rev_014", "text": "Fincher's direction in Fight Club is precise and suffocating. The unreliable-narrator twist holds up on every rewatch.",                                                                                                      "metadata": {"movie": "Fight Club",                          "year": 1999, "sentiment": "positive"}},
    {"id": "rev_015", "text": "The Matrix posed a genuinely philosophical question — what is real? — through the lens of cyberpunk action. Its bullet-time sequences still impress.",                                                                        "metadata": {"movie": "The Matrix",                          "year": 1999, "sentiment": "positive"}},
    {"id": "rev_016", "text": "Keanu Reeves's flat affect turns out to be perfect for Neo. He plays a man who has not yet woken up, in more ways than one.",                                                                                                "metadata": {"movie": "The Matrix",                          "year": 1999, "sentiment": "positive"}},
    {"id": "rev_017", "text": "Goodfellas is Scorsese at full throttle. The narration, the freeze-frames, the needle-drops — every technique serves the escalating paranoia.",                                                                               "metadata": {"movie": "Goodfellas",                          "year": 1990, "sentiment": "positive"}},
    {"id": "rev_018", "text": "Joe Pesci's Tommy DeVito is genuinely terrifying because his violence is random. Goodfellas never glamorises crime without showing its cost.",                                                                                 "metadata": {"movie": "Goodfellas",                          "year": 1990, "sentiment": "positive"}},
    {"id": "rev_019", "text": "Interstellar is emotionally ambitious in a way most sci-fi avoids. The docking sequence scored to Zimmer's pipe organ is extraordinary tension.",                                                                             "metadata": {"movie": "Interstellar",                        "year": 2014, "sentiment": "positive"}},
    {"id": "rev_020", "text": "Interstellar's third act polarises audiences — the tesseract sequence asks you to accept physics as metaphor. Not everyone makes that leap willingly.",                                                                        "metadata": {"movie": "Interstellar",                        "year": 2014, "sentiment": "mixed"}},
    {"id": "rev_021", "text": "Parasite is a perfectly calibrated class satire that pivots into a thriller without losing its dark wit. Bong Joon-ho makes it look effortless.",                                                                             "metadata": {"movie": "Parasite",                            "year": 2019, "sentiment": "positive"}},
    {"id": "rev_022", "text": "Parasite's architecture — both literal and structural — is the film's hidden genius. Every space tells you who has power and who does not.",                                                                                  "metadata": {"movie": "Parasite",                            "year": 2019, "sentiment": "positive"}},
    {"id": "rev_023", "text": "Whiplash is a film about the cost of greatness — and whether that cost is worth paying. J.K. Simmons's Fletcher is one of cinema's great antagonists.",                                                                      "metadata": {"movie": "Whiplash",                            "year": 2014, "sentiment": "positive"}},
    {"id": "rev_024", "text": "The final drumming sequence in Whiplash is one of the most exhilarating climaxes in recent cinema — pure kinetic energy, no dialogue needed.",                                                                               "metadata": {"movie": "Whiplash",                            "year": 2014, "sentiment": "positive"}},
    {"id": "rev_025", "text": "Dune achieves what previous adaptations could not — it makes Herbert's interiority legible through image and sound rather than expository dialogue.",                                                                          "metadata": {"movie": "Dune",                                "year": 2021, "sentiment": "positive"}},
    {"id": "rev_026", "text": "Villeneuve's Dune is deliberately paced. Some will find it cold; others will find it appropriately vast. It is a film about myth, not plot.",                                                                                  "metadata": {"movie": "Dune",                                "year": 2021, "sentiment": "mixed"}},
    {"id": "rev_027", "text": "Everything Everywhere All at Once is joyfully chaotic, but beneath the absurdism is a genuine meditation on generational trauma and immigrant identity.",                                                                      "metadata": {"movie": "Everything Everywhere All at Once",   "year": 2022, "sentiment": "positive"}},
    {"id": "rev_028", "text": "The Everything Bagel in EEAAO becomes one of cinema's most resonant symbols — nihilism as metaphor, defeated by ordinary love.",                                                                                             "metadata": {"movie": "Everything Everywhere All at Once",   "year": 2022, "sentiment": "positive"}},
    {"id": "rev_029", "text": "Oppenheimer is a moral horror film disguised as a biopic. Nolan lingers on the Trinity test not as triumph but as the birth of dread.",                                                                                      "metadata": {"movie": "Oppenheimer",                         "year": 2023, "sentiment": "mixed"}},
    {"id": "rev_030", "text": "Cillian Murphy's Oppenheimer is a man at war with his own mind. The non-linear structure mirrors his fractured relationship with his own legacy.",                                                                            "metadata": {"movie": "Oppenheimer",                         "year": 2023, "sentiment": "positive"}},
]                                                                # end of 30-chunk review list

with open("movie_reviews.json", "w") as f:                       # opens output file for writing
    json.dump(reviews, f, indent=2)                              # serialises list to pretty-printed JSON

print(f"✓ movie_reviews.json written ({len(reviews)} chunks)")   # confirms how many chunks were saved

✓ movie_reviews.json written (30 chunks)


In [ ]:
compose_step2 = """version: \"3.9\"

services:

  db:
    image: postgres:15-alpine
    restart: unless-stopped
    environment:
      POSTGRES_DB: ${POSTGRES_DB:-movies}
      POSTGRES_USER: ${POSTGRES_USER:-movie_user}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-secret}
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./seed_postgres.sql:/docker-entrypoint-initdb.d/seed.sql:ro

  chroma:                                # NEW — vector store service
    image: chromadb/chroma:latest        # official ChromaDB image
    restart: unless-stopped
    ports:
      - \"8000:8000\"                  # exposes Chroma HTTP API to the host
    volumes:
      - chromadata:/chroma/chroma        # named volume persists the embedding index
    environment:
      IS_PERSISTENT: \"TRUE\"           # enables on-disk persistence inside the container
      ANONYMIZED_TELEMETRY: \"FALSE\"   # disables Chroma telemetry data collection
    healthcheck:                         # NEW — Chroma healthcheck
      test: [\"CMD-SHELL\", \"bash -c 'echo > /dev/tcp/localhost/8000'\"]
      interval: 10s                      # probe runs every 10 seconds
      timeout: 5s                        # probe fails if no response within 5 seconds
      retries: 5                         # container marked unhealthy after 5 consecutive failures
      start_period: 15s                  # grace period before retries count (Chroma index load time)

  api:
    build: ./api
    restart: unless-stopped
    ports:
      - \"8080:8080\"
    environment:
      DATABASE_URL: postgresql://${POSTGRES_USER:-movie_user}:${POSTGRES_PASSWORD:-secret}@db:5432/${POSTGRES_DB:-movies}
      CHROMA_HOST: chroma
      CHROMA_PORT: \"8000\"
    depends_on:                          # api now waits for chroma to pass its healthcheck
      chroma:
        condition: service_healthy       # 'service_healthy' requires a passing healthcheck probe

volumes:
  pgdata:
  chromadata:                            # NEW — named volume for Chroma embedding index
"""

with open("docker-compose.yml", "w") as f:           # overwrites previous Step-1 file
    f.write(compose_step2)                            # writes Step-2 Compose definition

run("docker compose up -d")                           # docker compose detects the new chroma service and starts it

print("\n✓ Chroma service added. Watch it become healthy:")      # prompts student to observe healthcheck
print("  docker compose ps  (run in a separate terminal)")       # shows command to check container status

## Beat 4 — Adding the Frontend Service (12 min)

We add the `web` service (Next.js) to the Compose file.

### Pre-identified artifact — slide: *build-time vs runtime env in Next.js*

| Variable prefix | When it is evaluated | How to change it |
|-----------------|---------------------|------------------|
| `NEXT_PUBLIC_*` | At **build time** (`npm run build`) | Must rebuild the image |
| Server-only `process.env.*` | At **runtime** (each server render) | Change env var and restart |

**Consequence:** If you set `NEXT_PUBLIC_API_URL` in `docker compose up -e`, the running container does **not** pick it up — the value is already baked into the JavaScript bundle.  
The Dockerfile passes it in via `ARG → ENV` at build time.

```
docker compose build --build-arg NEXT_PUBLIC_API_URL=https://prod-api.example.com web
docker compose up -d web
```

### Dependency
`web` depends on `api` with `condition: service_healthy`.  
This prevents the browser from loading the UI before the API is reachable.

In [10]:
%%writefile web/package.json
{
  "name": "movie-web",
  "version": "0.1.0",
  "private": true,
  "scripts": {
    "dev":   "next dev",
    "build": "next build",
    "start": "next start -p 3000"
  },
  "dependencies": {
    "next":      "14.2.3",
    "react":     "18.3.1",
    "react-dom": "18.3.1"
  }
}

Overwriting web/package.json


In [11]:
%%writefile web/pages/index.js
// Movie Review Search — Next.js page component
import { useState } from 'react'                                 // useState hook for local component state

const API_URL = process.env.NEXT_PUBLIC_API_URL || 'http://localhost:8080'  // baked at build time

export default function Home() {
  const [query,   setQuery]   = useState('')                     // controlled input value
  const [results, setResults] = useState(null)                   // search result payload from the API
  const [error,   setError]   = useState(null)                   // error message to display
  const [loading, setLoading] = useState(false)                  // disables the button while fetching

  async function handleSearch(e) {
    e.preventDefault()                                           // prevents page reload on form submit
    setLoading(true)                                             // shows loading state immediately
    setError(null)                                               // clears any previous error
    try {
      const res = await fetch(`${API_URL}/search?q=${encodeURIComponent(query)}`)  // calls the RAG endpoint
      if (!res.ok) {                                             // non-2xx status means an API error
        const body = await res.json()                            // parses JSON error detail
        throw new Error(body.detail || `HTTP ${res.status}`)    // surfaces the error message
      }
      setResults(await res.json())                               // stores successful response in state
    } catch (err) {
      setError(err.message)                                      // stores error text for display
    } finally {
      setLoading(false)                                          // always re-enables the button
    }
  }

  return (
    <main style={{ maxWidth: 800, margin: '2rem auto', fontFamily: 'sans-serif' }}>
      <h1>Movie Review Search</h1>
      <form onSubmit={handleSearch}>
        <input
          value={query}
          onChange={e => setQuery(e.target.value)}
          placeholder="e.g. hopeful prison drama"               /* placeholder guides the user */
          style={{ width: '70%', padding: '0.5rem' }}
        />
        <button type="submit" disabled={loading} style={{ padding: '0.5rem 1rem', marginLeft: 8 }}>
          {loading ? 'Searching\u2026' : 'Search'}              /* shows spinner text while fetching */
        </button>
      </form>

      {error && (                                                /* renders error banner when error exists */
        <div style={{ color: 'red', marginTop: '1rem' }}>
          <strong>Error:</strong> {error}                        /* displays the error detail from the API */
        </div>
      )}

      {results && (                                              /* renders results list when data exists */
        <div style={{ marginTop: '1.5rem' }}>
          <h2>Results for \u201c{results.query}\u201d</h2>
          {results.results.map((text, i) => (                   /* maps each result chunk to a card */
            <div key={i} style={{ border: '1px solid #ddd', padding: '1rem', marginBottom: '1rem', borderRadius: 4 }}>
              <p>{text}</p>
              <small style={{ color: '#666' }}>
                {results.metadatas[i].movie} ({results.metadatas[i].year}) \u2014 {results.metadatas[i].sentiment}
              </small>
            </div>
          ))}
        </div>
      )}
    </main>
  )
}

Overwriting web/pages/index.js


In [12]:
%%writefile web/Dockerfile
# ── Stage 1: build ───────────────────────────────────────────────
# Node 18 on Alpine as the build environment
FROM node:18-alpine AS builder
WORKDIR /app
# copies package.json and package-lock.json for cache efficiency
COPY package*.json ./
RUN npm ci                              # installs exact locked dependencies (faster than npm install)
# copies all source files including pages/
COPY . .

# build-time argument — value is baked into the JS bundle
ARG NEXT_PUBLIC_API_URL=http://localhost:8080
# promotes ARG into ENV so Next.js picks it up at build
ENV NEXT_PUBLIC_API_URL=$NEXT_PUBLIC_API_URL

RUN npm run build                       # runs 'next build' — compiles pages, inlines NEXT_PUBLIC vars

# ── Stage 2: production runtime ──────────────────────────────────
# fresh minimal image for the production runtime layer
FROM node:18-alpine AS runner
WORKDIR /app
# tells Next.js to run in production mode (smaller, faster)
ENV NODE_ENV production
# copies the compiled Next.js output from the builder stage
COPY --from=builder /app/.next ./.next
# copies installed modules
COPY --from=builder /app/node_modules ./node_modules
# copies package.json for npm start
COPY --from=builder /app/package.json ./
# documents the port Next.js listens on
EXPOSE 3000
# starts the Next.js production server
CMD ["npm", "start"]

Overwriting web/Dockerfile


## Beat 5 — Healthcheck + `depends_on` Chain (15 min)

### The failure mode (pre-identified artifact — terminal capture from rehearsal)

Without `condition: service_healthy`, Compose starts all services in dependency order but does **not**
wait for them to be *ready*. The sequence you observe:

```
t=0   db container starts
t=0   chroma container starts
t=1   api container starts (Postgres is still running initdb scripts)
t=2   api receives first request → psycopg2.connect() → Connection refused → 500
t=10  Postgres finishes initdb — too late, API already crashed
```

### The fix — full `depends_on` chain

```
web  ─── service_healthy ──► api
api  ─── service_healthy ──► db
api  ─── service_healthy ──► chroma
```

Each arrow means: "do not start me until the target container's healthcheck returns healthy."

### Healthcheck commands per service

| Service | Healthcheck command |
|---------|--------------------|
| `db` | `pg_isready -U <user> -d <db>` |
| `chroma` | `curl -sf http://localhost:8000/api/v1/heartbeat` |
| `api` | `curl -sf http://localhost:8080/readyz` |

> **Team note:** For the assignment stack (Neo4j + Weaviate), the healthcheck commands differ:  
> Neo4j → `cypher-shell -u neo4j -p $NEO4J_PASSWORD 'RETURN 1'`  
> Weaviate → `curl -sf http://localhost:8080/v1/.well-known/ready`

In [ ]:
%%writefile docker-compose.yml
version: "3.9"                        # Compose file-format version

services:

  # ──────────────────────────────────────────────────────────────
  # db — PostgreSQL relational store
  # ──────────────────────────────────────────────────────────────
  db:
    image: postgres:15-alpine          # official Postgres 15 on Alpine (lightweight)
    restart: unless-stopped            # auto-restarts on crash; stops only on explicit docker compose stop
    environment:
      POSTGRES_DB:       ${POSTGRES_DB:-movies}           # database name; default 'movies'
      POSTGRES_USER:     ${POSTGRES_USER:-movie_user}     # db user; default 'movie_user'
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-secret}     # password; set in .env for production
    volumes:
      - pgdata:/var/lib/postgresql/data                   # named volume — data survives down
      - ./seed_postgres.sql:/docker-entrypoint-initdb.d/seed.sql:ro  # runs once on first boot
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U ${POSTGRES_USER:-movie_user} -d ${POSTGRES_DB:-movies}"]
      interval:     10s                # probe runs every 10 seconds
      timeout:       5s                # probe times out after 5 seconds
      retries:        5                # marked unhealthy after 5 consecutive failures
      start_period: 10s                # ignores failures during the first 10 s (initdb grace period)

  # ──────────────────────────────────────────────────────────────
  # chroma — ChromaDB vector store
  # ──────────────────────────────────────────────────────────────
  chroma:
    image: chromadb/chroma:latest      # official ChromaDB image
    restart: unless-stopped
    ports:
      - "8000:8000"                    # exposes Chroma HTTP API to host
    volumes:
      - chromadata:/chroma/chroma      # named volume persists the embedding index
    environment:
      IS_PERSISTENT:        "TRUE"     # tells Chroma to persist embeddings to disk
      ANONYMIZED_TELEMETRY: "FALSE"    # disables telemetry data collection
    healthcheck:
      test: ["CMD-SHELL", "bash -c 'echo > /dev/tcp/localhost/8000'"]
      interval:     10s
      timeout:       5s
      retries:        5
      start_period: 15s                # Chroma needs extra time to load its index on startup

  # ──────────────────────────────────────────────────────────────
  # api — FastAPI backend
  # ──────────────────────────────────────────────────────────────
  api:
    build: ./api                       # builds image from the Dockerfile in ./api
    restart: unless-stopped
    ports:
      - "8080:8080"                    # exposes FastAPI app port to host
    environment:
      DATABASE_URL: postgresql://${POSTGRES_USER:-movie_user}:${POSTGRES_PASSWORD:-secret}@db:5432/${POSTGRES_DB:-movies}
      CHROMA_HOST:  chroma             # Docker network resolves service name 'chroma' to its IP
      CHROMA_PORT:  "8000"
    depends_on:
      db:                              # api will not start until db passes its healthcheck
        condition: service_healthy
      chroma:                          # api will not start until chroma passes its healthcheck
        condition: service_healthy
    healthcheck:
      test: ["CMD-SHELL", "curl -sf http://localhost:8080/readyz || exit 1"]
      interval:     10s
      timeout:       5s
      retries:        5
      start_period: 20s                # FastAPI needs time after both dependencies are ready

  # ──────────────────────────────────────────────────────────────
  # web — Next.js frontend
  # ──────────────────────────────────────────────────────────────
  web:
    build: ./web                       # builds image from the Dockerfile in ./web
    restart: unless-stopped
    ports:
      - "3000:3000"                    # exposes Next.js server port to host
    environment:
      NEXT_PUBLIC_API_URL: ${NEXT_PUBLIC_API_URL:-http://localhost:8080}  # baked at build time
    depends_on:
      api:                             # web will not start until api passes its healthcheck
        condition: service_healthy

# ────────────────────────────────────────────────────────────────
# Named volumes — survive 'docker compose down'; removed with -v
# ────────────────────────────────────────────────────────────────
volumes:
  pgdata:      # PostgreSQL data directory
  chromadata:  # ChromaDB embedding index

In [14]:
print("Building and starting the full four-service stack...")    # informs user of what is about to happen
print("This takes 60–120 s while images build and healthchecks pass.\n")  # sets time expectation

result = run("docker compose up --build -d", capture=False)      # builds images and starts all services detached

print("\nWaiting 30 s for all healthchecks to converge...")       # prompts student to be patient
time.sleep(30)                                                   # gives containers time to become healthy

run("docker compose ps")                                          # shows current status of all four containers

print("\n✓ All services started. Expected STATUS: healthy or running.")

Building and starting the full four-service stack...
This takes 60–120 s while images build and healthchecks pass.

$ docker compose up --build -d


time="2026-06-25T05:54:40+03:00" level=warning msg="/home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 10A Integrated Lab/docker-compose.yml: `version` is obsolete"


#0 building with "default" instance using docker driver

#1 [api internal] load build definition from Dockerfile
#1 transferring dockerfile:
#1 transferring dockerfile: 764B done
#1 DONE 1.4s

#2 [api internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 7.2s

#3 [api internal] load .dockerignore
#3 transferring context:
#3 transferring context: 2B done
#3 DONE 1.3s

#4 [api 1/5] FROM docker.io/library/python:3.11-slim@sha256:cdbd05fb6f457ca275ff51ce00d93d865ca0b6a25f5ffb08262d94f6835771e5
#4 resolve docker.io/library/python:3.11-slim@sha256:cdbd05fb6f457ca275ff51ce00d93d865ca0b6a25f5ffb08262d94f6835771e5
#4 ...

#5 [api internal] load build context
#5 transferring context: 94B done
#5 DONE 1.4s

#4 [api 1/5] FROM docker.io/library/python:3.11-slim@sha256:cdbd05fb6f457ca275ff51ce00d93d865ca0b6a25f5ffb08262d94f6835771e5
#4 resolve docker.io/library/python:3.11-slim@sha256:cdbd05fb6f457ca275ff51ce00d93d865ca0b6a25f5ffb08262d94f6835771e5 2.6s done
#4 DONE 2.6s

#6 [api 4/

 Container module10aintegratedlab-db-1  Recreate
 Container module10aintegratedlab-chroma-1  Running
 Container module10aintegratedlab-db-1  Recreated
 Container module10aintegratedlab-api-1  Recreate
 Container module10aintegratedlab-api-1  Recreated
 Container module10aintegratedlab-web-1  Recreate
 Container module10aintegratedlab-web-1  Recreated
 Container module10aintegratedlab-db-1  Starting
Error response from daemon: failed to set up container networking: driver failed programming external connectivity on endpoint module10aintegratedlab-db-1 (fac70e8037453b589354d757e7ddf49129458adb7612ed0bbe21a4ddc68d0c1d): Bind for 0.0.0.0:5432 failed: port is already allocated



Waiting 30 s for all healthchecks to converge...
$ docker compose ps
NAME                              IMAGE                    COMMAND                  SERVICE   CREATED         STATUS                   PORTS
module10aintegratedlab-chroma-1   chromadb/chroma:latest   "dumb-init -- chroma…"   chroma    3 minutes ago   Up 2 minutes (healthy)   0.0.0.0:8000->8000/tcp, :::8000->8000/tcp

✓ All services started. Expected STATUS: healthy or running.


## Beat 6 — `.env` Discipline (8 min)

### Why `.env` files must never be committed
- They contain secrets (passwords, API keys, connection strings)
- Once committed, secrets are in git history **forever** — rotating the value is not enough
- CI/CD systems inject secrets via environment variables or secret managers — not files

### Compose env substitution
When Compose sees `${VAR:-default}` in a `docker-compose.yml`, it:
1. Reads `VAR` from the shell environment
2. Falls back to a `.env` file in the same directory
3. Uses `default` if neither is set

`.env.example` documents what variables must be set — commit it, not `.env`.

### Pre-identified artifact — `.env` scanner script
The scanner below detects committed `.env` files before a push.

In [15]:
%%writefile .env.example
# Copy this file to .env and fill in real values.
# NEVER commit .env — it is listed in .gitignore.

# PostgreSQL ──────────────────────────────────────
POSTGRES_DB=movies                   # name of the database to create
POSTGRES_USER=movie_user             # name of the database user
POSTGRES_PASSWORD=CHANGE_ME          # strong password — replace before deploying

# Next.js (baked at build time) ───────────────────
NEXT_PUBLIC_API_URL=http://localhost:8080  # URL the browser uses to reach the API

Overwriting .env.example


In [16]:
%%writefile .gitignore
# Secrets — never commit these
.env
.env.local
.env.production

# Python
__pycache__/
*.pyc
.venv/

# Node
node_modules/
web/.next/

# Docker artefacts
*.log

Overwriting .gitignore


In [17]:
import pathlib                                                    # provides object-oriented filesystem paths

def scan_committed_env_files(repo_root="."):                     # scans a repo for accidentally committed .env files
    root = pathlib.Path(repo_root)                               # converts the string path to a Path object
    result = subprocess.run(                                      # runs git to list all tracked files
        "git ls-files",                                          # lists files known to the index (committed)
        shell=True, capture_output=True, text=True, cwd=root     # runs in the given directory
    )
    tracked = result.stdout.splitlines()                          # splits output into one filename per line
    violations = [                                                # filters for .env files in the tracked list
        f for f in tracked                                        # iterates over every tracked filename
        if pathlib.Path(f).name in (".env", ".env.local", ".env.production")  # matches secret filenames
    ]
    return violations                                             # returns list of violations (empty if clean)

violations = scan_committed_env_files()                          # runs the scan on the current repo

if violations:                                                   # at least one .env file is committed
    print("✗ COMMITTED .env FILES FOUND — ROTATE SECRETS IMMEDIATELY:")  # urgent warning
    for v in violations:                                         # iterates over each violation
        print(f"  git rm --cached {v}")                          # shows the command to untrack the file
else:                                                            # no violations — repo is clean
    print("✓ No committed .env files found.")                    # confirms the repo passes the check

print("\n.env.example committed? ",                             # checks if the example file is tracked
      ".env.example" in subprocess.run("git ls-files", shell=True, capture_output=True, text=True).stdout)

✓ No committed .env files found.

.env.example committed?  False


## Beat 7 — Seed Orchestration (15 min)

### Why order matters
1. **Postgres** must be seeded before the API starts serving `/movies` — but the `initdb.d/` mount
   handles this automatically on first boot.
2. **Chroma** must be seeded manually after the Chroma container is healthy — there is no built-in
   auto-seed mechanism for vector stores.

### Idempotency
Both seed scripts are idempotent:
- `seed_postgres.sql` uses `ON CONFLICT DO NOTHING`
- `seed_chroma.py` uses `get_or_create_collection` and skips IDs that already exist

Running them twice produces the same result.

### Pre-identified artifacts
- `seed_postgres.sh` — shell wrapper that calls `psql` inside the `db` container
- `seed_chroma.py` — Python script that uses the ChromaDB client to upsert 30 chunks

In [18]:
%%writefile seed_postgres.sh
#!/usr/bin/env bash
# seed_postgres.sh — runs the SQL seed file inside the running db container
# Usage: ./seed_postgres.sh
set -euo pipefail                        # exit on error; treat unset vars as errors; fail on pipe errors

CONTAINER=$(docker compose ps -q db)     # gets the container ID of the db service

if [ -z "$CONTAINER" ]; then             # checks whether the db container is running
  echo "Error: db container is not running. Run 'docker compose up -d db' first."
  exit 1                                 # exits with a non-zero code to signal failure
fi

echo "Seeding Postgres..."               # progress message

docker exec -i "$CONTAINER" \            # runs a command inside the container
  psql -U "${POSTGRES_USER:-movie_user}" \  # connects as the movie_user (reads from env or uses default)
       -d "${POSTGRES_DB:-movies}" \        # targets the movies database
  < seed_postgres.sql                    # pipes the SQL file into psql's stdin

echo "✓ Postgres seed complete."         # confirms the seed ran without error

Overwriting seed_postgres.sh


In [19]:
%%writefile seed_chroma.py
#!/usr/bin/env python3
"""seed_chroma.py — loads movie_reviews.json into the Chroma movie_reviews collection."""
import importlib.metadata, subprocess, sys, os

# Chroma server 1.4.4+ dropped the v1 HTTP API; the client must be ≥1.4 to speak v2.
_ver = tuple(int(x) for x in importlib.metadata.version("chromadb").split(".")[:2])
if _ver < (1, 4):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "chromadb>=1.4.1"])
    os.execv(sys.executable, [sys.executable] + sys.argv)   # re-exec with upgraded package

import chromadb                                                   # ChromaDB Python client
import json                                                       # reads the JSON seed file
import sys                                                        # used for sys.exit on error

CHROMA_HOST = "localhost"                                         # connects to the published host port
CHROMA_PORT = 8000                                                # Chroma's default HTTP port
SEED_FILE   = "movie_reviews.json"                               # path to the pre-built reviews JSON
COLLECTION  = "movie_reviews"                                    # collection name the API queries

client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT) # creates an HTTP client to the Chroma service

try:                                                              # verifies Chroma is reachable
    client.heartbeat()                                            # raises an exception if Chroma is down
except Exception as exc:                                          # catches connection failures
    print(f"Cannot reach Chroma at {CHROMA_HOST}:{CHROMA_PORT}: {exc}")  # reports the error
    sys.exit(1)                                                   # exits with non-zero code to signal failure

collection = client.get_or_create_collection(COLLECTION)         # creates the collection if it does not exist

with open(SEED_FILE) as f:                                        # opens the pre-built reviews file
    reviews = json.load(f)                                        # parses the JSON list into Python objects

existing = set(collection.get()["ids"])                          # fetches IDs already in the collection

new_reviews = [r for r in reviews if r["id"] not in existing]   # filters out already-seeded chunks (idempotent)

if not new_reviews:                                              # skips the upsert if all chunks already exist
    print(f"✓ Collection '{COLLECTION}' already has all {len(reviews)} chunks — nothing to do.")
    sys.exit(0)                                                   # exits cleanly

collection.add(                                                   # adds new chunks to the collection
    ids=[r["id"] for r in new_reviews],                          # unique ID per chunk
    documents=[r["text"] for r in new_reviews],                  # raw text to embed and store
    metadatas=[r["metadata"] for r in new_reviews]               # metadata dict returned with query results
)

print(f"✓ Seeded {len(new_reviews)} new chunks into '{COLLECTION}' (total: {len(reviews)}).")  # summary

Overwriting seed_chroma.py


In [20]:
print("=== Seed Orchestration ===")                              # section header

print("\n[1/2] Postgres seed (via initdb.d — runs automatically on first boot)")  # explains automatic seeding
r = run("docker compose exec db psql -U movie_user -d movies -c 'SELECT COUNT(*) FROM movies;'")  # counts seeded rows
print("      ^ If count = 20, Postgres was already seeded via initdb.d.")         # contextualises the output

print("\n[2/2] Chroma seed...")                                   # announces the Chroma seeding step
r = run("python3 seed_chroma.py")                                # runs the Python seed script
if r.returncode != 0:                                            # checks whether the script succeeded
    print("✗ Chroma seed failed — is Chroma healthy?")            # warns if seeding failed
    print("  Run: docker compose ps")                             # suggests checking container status
else:                                                            # seed succeeded
    print("✓ Both datastores seeded.")                            # confirms both seeds completed

=== Seed Orchestration ===

[1/2] Postgres seed (via initdb.d — runs automatically on first boot)
$ docker compose exec db psql -U movie_user -d movies -c 'SELECT COUNT(*) FROM movies;'


time="2026-06-25T05:57:03+03:00" level=warning msg="/home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 10A Integrated Lab/docker-compose.yml: `version` is obsolete"
service "db" is not running


      ^ If count = 20, Postgres was already seeded via initdb.d.

[2/2] Chroma seed...
$ python3 seed_chroma.py
✓ Seeded 30 new chunks into 'movie_reviews' (total: 30).
✓ Both datastores seeded.


## Beat 8 — End-to-End Demo from the Browser (10 min)

### Happy path
1. Open `http://localhost:3000` in a browser
2. Type a query such as *"hopeful prison drama"*
3. Observe review chunks with movie citations returned below the search bar

### Partial-failure recovery demo
This beat demonstrates what the system does when Chroma goes down mid-session.

| Step | Action | Expected observation |
|------|--------|---------------------|
| 1 | Search works | 3 results with citations |
| 2 | `docker compose stop chroma` | `/readyz` returns 503 |
| 3 | Submit a search | Frontend shows `Error: Chroma not ready` |
| 4 | `docker compose start chroma` | Chroma restarts; healthcheck passes |
| 5 | `/readyz` returns 200 | Frontend search works again |

This mirrors real-world partial-failure scenarios and shows that the `/readyz` probe correctly reports dependency health.

In [21]:
import urllib.request                                             # built-in HTTP client for simple GET requests
import urllib.error                                              # provides URLError for catching HTTP errors

def http_get(url):                                               # sends a GET request and returns (status, body)
    try:                                                         # attempts the HTTP request
        with urllib.request.urlopen(url, timeout=5) as resp:    # opens the URL with a 5-second timeout
            return resp.status, resp.read().decode()            # returns status code and decoded response body
    except urllib.error.HTTPError as e:                          # catches HTTP error responses (4xx, 5xx)
        return e.code, e.read().decode()                         # returns the error status and body
    except Exception as e:                                       # catches connection refused, timeout, etc.
        return 0, str(e)                                         # returns 0 and the error message string

print("=== Stack Health Check ===")                             # section header

status, body = http_get("http://localhost:8080/readyz")          # probes the API readiness endpoint
print(f"  /readyz        → {status}  {body[:60]}")              # prints status code and first 60 chars

status, body = http_get("http://localhost:8080/movies")          # fetches the movie list from Postgres via API
movies = json.loads(body) if status == 200 else []               # parses JSON only on success
print(f"  /movies        → {status}  ({len(movies)} movies)")   # shows how many movies were returned

status, body = http_get("http://localhost:8080/search?q=hopeful+prison+drama")  # runs a RAG search query
print(f"  /search?q=...  → {status}  {body[:80]}")              # prints first 80 chars of search result

print("\nOpen http://localhost:3000 in a browser to test the full UI.")

=== Stack Health Check ===
  /readyz        → 0  <urlopen error [Errno 111] Connection refused>
  /movies        → 0  (0 movies)
  /search?q=...  → 0  <urlopen error [Errno 111] Connection refused>

Open http://localhost:3000 in a browser to test the full UI.


In [22]:
print("=== Partial-Failure Demo ===")                            # section header

print("\n[Step 1] Stopping Chroma mid-session...")               # announces Step 1
run("docker compose stop chroma")                                # stops only the chroma service
time.sleep(5)                                                    # waits for healthcheck to detect the failure

print("[Step 2] Probing /readyz — expect 503...")                # announces readiness probe step
status, body = http_get("http://localhost:8080/readyz")          # probes the API readiness endpoint
print(f"  /readyz → {status}  {body}")                          # should show 503 + Chroma not ready message

print("[Step 3] Probing /search — expect 503...")                # announces search probe step
status, body = http_get("http://localhost:8080/search?q=drama") # runs a search while Chroma is down
print(f"  /search → {status}  {body[:120]}")                    # should show 503 error response

print("\n[Step 4] Restarting Chroma...")                         # announces recovery step
run("docker compose start chroma")                               # restarts the chroma service
print("Waiting 20 s for Chroma to become healthy again...")      # sets expectation for recovery time
time.sleep(20)                                                   # waits for healthcheck to pass

print("[Step 5] Probing /readyz — expect 200...")                # announces recovery verification
status, body = http_get("http://localhost:8080/readyz")          # re-probes the readiness endpoint
print(f"  /readyz → {status}  {body}")                          # should show 200 + ok

print("\n✓ Partial-failure recovery demonstrated.")              # confirms the demo completed

=== Partial-Failure Demo ===

[Step 1] Stopping Chroma mid-session...
$ docker compose stop chroma
[Step 2] Probing /readyz — expect 503...
  /readyz → 0  <urlopen error [Errno 111] Connection refused>
[Step 3] Probing /search — expect 503...
  /search → 0  <urlopen error [Errno 111] Connection refused>

[Step 4] Restarting Chroma...
$ docker compose start chroma
Waiting 20 s for Chroma to become healthy again...
[Step 5] Probing /readyz — expect 200...
  /readyz → 0  <urlopen error [Errno 111] Connection refused>

✓ Partial-failure recovery demonstrated.


## Beat 9 — Runbook Authoring (10 min)

A runbook answers: *"What does a new engineer need to know to bring this stack up, verify it works,
and recover from common failures?"*

We write it live as if reading it aloud to someone onboarding remotely.

In [23]:
%%writefile README.md
# Movie Information Service — Operations Runbook

## Overview

Four-service Docker Compose stack: FastAPI + PostgreSQL + ChromaDB + Next.js.

| Service | Port | Purpose |
|---------|------|---------|
| `web`   | 3000 | Next.js frontend |
| `api`   | 8080 | FastAPI backend |
| `db`    | 5432 | PostgreSQL movie catalog |
| `chroma`| 8000 | ChromaDB review embeddings |

---

## Prerequisites

- Docker Desktop ≥ 4.x with ≥ 8 GB RAM allocated
- Docker Compose v2 (`docker compose version`)
- Python 3.11+ (for the Chroma seed script)
- `pip install chromadb` (for local seed script execution)

---

## First-time setup

```bash
# 1. Clone the repository
git clone <repo-url> && cd <repo-dir>

# 2. Create your .env file from the example
cp .env.example .env
#    Edit .env and set POSTGRES_PASSWORD to a real secret

# 3. Pull images that are not built locally
docker pull postgres:15-alpine
docker pull chromadb/chroma:latest

# 4. Build and start all services
docker compose up --build -d

# 5. Verify all services are healthy (wait ~60 s)
docker compose ps

# 6. Seed Chroma with movie-review embeddings
python seed_chroma.py

# 7. Open the UI
#    http://localhost:3000
```

---

## Day-to-day commands

| Task | Command |
|------|---------|
| Start all services | `docker compose up -d` |
| Stop all services (keep data) | `docker compose down` |
| Stop and wipe volumes | `docker compose down -v` |
| Rebuild one service | `docker compose up --build -d api` |
| Follow all logs | `docker compose logs -f` |
| Follow one service | `docker compose logs -f api` |
| Check health | `docker compose ps` |
| Open a psql shell | `docker compose exec db psql -U movie_user -d movies` |

---

## Readiness probe

```bash
curl http://localhost:8080/readyz
# 200 {"status": "ok"}       — all dependencies healthy
# 503 {"detail": "..."}      — Postgres or Chroma is not reachable
```

---

## Common failure scenarios

### API starts but immediately exits
Most likely Postgres or Chroma was not healthy when the API started.  
Check: `docker compose logs api`  
Fix: `docker compose restart api` (after the dependency is healthy)

### `/search` returns 503 — Chroma not ready
Chroma may have crashed or its index is loading.  
Check: `docker compose ps chroma`  
Fix: `docker compose restart chroma` then wait ~15 s

### Postgres seed is empty
The `initdb.d/` directory only runs on first boot. If pgdata volume already exists, the SQL file is skipped.  
Fix: `docker compose down -v && docker compose up -d` to recreate volumes.

### Next.js shows wrong API URL
`NEXT_PUBLIC_API_URL` is baked at build time. Changing `.env` and restarting is not enough.  
Fix: `docker compose up --build -d web`

---

## Checking for committed secrets

```bash
git ls-files | grep -E '^\.env$|\.env\.local$|\.env\.production$'
# Should return nothing. If it returns a path, run:
# git rm --cached .env && git commit -m "remove secret file"
```

---

## Architecture diagram

```
browser → web:3000 → api:8080 → db:5432   (movie catalog)
                              → chroma:8000 (review embeddings)
```

Overwriting README.md


## Beat 10 — Team Rollout & Assignment Pivot (10 min)

### What changes for your team

Your team assignment uses **Neo4j** (instead of Postgres) and **Weaviate** (instead of Chroma).
The orchestration discipline — depends_on, healthchecks, volume discipline, env-scope discipline — is **identical**.
Only the images, ports, and healthcheck commands differ:

| Concept | SI demo (this lab) | Team assignment |
|---------|-------------------|----------------|
| Relational / graph store | `postgres:15-alpine` port 5432 | `neo4j:5` port 7474/7687 |
| Vector store | `chromadb/chroma:latest` port 8000 | `semitechnologies/weaviate` port 8080 |
| DB healthcheck | `pg_isready` | `cypher-shell -u neo4j -p $NEO4J_PASSWORD 'RETURN 1'` |
| Vector healthcheck | `curl /api/v1/heartbeat` | `curl /v1/.well-known/ready` |
| Postgres seed | `.sql` via `initdb.d/` | Cypher (`.cypher`) via `neo4j-admin import` or driver |
| Chroma seed | Python `chromadb` client | Python `weaviate-client` library |

---

### Team workflow recap

1. **One team fork** from `m10-i10` — team submitter creates it
2. **Three role branches** — Backend lead, Frontend lead, DevOps lead each cut a branch
3. **Internal PRs** — each role's branch opens a PR reviewed by the teammate whose contract surface the change touches
4. **Local smoke test** — `docker compose up -d` on the merged fork passes before submitting
5. **TalentLMS** — team submitter pastes the team fork URL; each team member submits a participation checkbox

---

### Contract discipline (do not skip this)

- If you **rename a Pydantic field** → announce it on Slack *before* the change lands
- If you **change `NEXT_PUBLIC_API_URL`** → announce it on Slack *before* the change lands
- If Frontend lead needs a new backend field → open an **internal PR comment** on the Backend lead's branch — do not assume

---

### Escalation path

> Inline PR comment → team Slack with TA tagged → Support Instructor → Lead Instructor

Open the Integration Guide; the autograder brings the full four-service stack up in CI.